# 02e — Feature Engineering

**"Garbage in, garbage out."**  
The best model in the world can't save bad features. Good features > complex model.

```
Raw Data ──► Feature Engineering ──► Clean Features ──► Model ──► Good Predictions
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing, load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, LabelEncoder, 
    OrdinalEncoder, PolynomialFeatures
)
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import mutual_info_classif, RFE
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, r2_score

plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['figure.dpi'] = 100
np.random.seed(42)

---
## 1 — Numerical Features: Scaling

Many algorithms (KNN, SVM, logistic regression, neural networks) are **sensitive to feature scale**.  
A feature ranging 0-1000 will dominate one ranging 0-1.

| Scaler | Formula | Range | When |
|--------|---------|-------|------|
| StandardScaler | $(x - \mu) / \sigma$ | mean=0, std=1 | Most algorithms |
| MinMaxScaler | $(x - \min) / (\max - \min)$ | [0, 1] | Neural networks, images |

In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sample = df[['MedInc', 'AveRooms', 'Population']].values[:200]

axes[0].boxplot(sample, labels=['MedInc', 'AveRooms', 'Population'])
axes[0].set_title('Before scaling — wildly different ranges')

standard = StandardScaler().fit_transform(sample)
axes[1].boxplot(standard, labels=['MedInc', 'AveRooms', 'Population'])
axes[1].set_title('StandardScaler — mean=0, std=1')

minmax = MinMaxScaler().fit_transform(sample)
axes[2].boxplot(minmax, labels=['MedInc', 'AveRooms', 'Population'])
axes[2].set_title('MinMaxScaler — [0, 1]')

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error

X = df.drop('MedHouseVal', axis=1)
y = df['MedHouseVal']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

knn_raw = KNeighborsRegressor().fit(X_tr, y_tr)
r2_raw = knn_raw.score(X_te, y_te)

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)
knn_scaled = KNeighborsRegressor().fit(X_tr_s, y_tr)
r2_scaled = knn_scaled.score(X_te_s, y_te)

print(f'KNN without scaling: R² = {r2_raw:.4f}')
print(f'KNN with scaling:    R² = {r2_scaled:.4f}')
print(f'Improvement:         {r2_scaled - r2_raw:+.4f}')

---
## 2 — Categorical Features: Encoding

Models need numbers. Categorical features need to be converted.

| Method | When to use | Example |
|--------|-----------|----------|
| **One-Hot Encoding** | Nominal (no order): color, city | color → [is_red, is_blue, is_green] |
| **Label Encoding** | Tree-based models can handle it | red=0, blue=1, green=2 |
| **Ordinal Encoding** | Ordered categories | small=0, medium=1, large=2 |

In [ ]:
sample_df = pd.DataFrame({
    'color': ['red', 'blue', 'green', 'blue', 'red', 'green'],
    'size': ['S', 'M', 'L', 'M', 'S', 'L'],
    'price': [10, 20, 30, 25, 15, 35],
})

print("Original:")
print(sample_df)
print()

onehot = pd.get_dummies(sample_df, columns=['color'], drop_first=True)
print("One-Hot Encoded (drop_first to avoid multicollinearity):")
print(onehot)
print()

le = LabelEncoder()
sample_df['color_label'] = le.fit_transform(sample_df['color'])
print(f"Label Encoded: {dict(zip(le.classes_, le.transform(le.classes_)))}")

oe = OrdinalEncoder(categories=[['S', 'M', 'L']])
sample_df['size_ordinal'] = oe.fit_transform(sample_df[['size']])
print(f"Ordinal Encoded: S=0, M=1, L=2")
print(sample_df)

---
## 3 — Missing Values

Real data is messy. Missing values are everywhere.

| Strategy | When | Pros | Cons |
|----------|------|------|------|
| Drop rows | Very few missing | Simple | Lose data |
| Mean/Median | Numerical, random missing | Preserves distribution | Ignores correlations |
| Mode | Categorical | Simple | May bias |
| Indicator column | Missingness is informative | Captures pattern | Extra feature |

In [ ]:
np.random.seed(42)
df_messy = pd.DataFrame({
    'age': [25, np.nan, 35, 40, np.nan, 30, 45, np.nan, 28, 50],
    'income': [50000, 60000, np.nan, 80000, 55000, np.nan, 90000, 45000, 65000, np.nan],
    'category': ['A', 'B', 'A', np.nan, 'B', 'A', np.nan, 'B', 'A', 'B'],
})

print("Missing values:")
print(df_messy.isnull().sum())
print()

imp_median = SimpleImputer(strategy='median')
df_messy[['age', 'income']] = imp_median.fit_transform(df_messy[['age', 'income']])

imp_mode = SimpleImputer(strategy='most_frequent')
df_messy[['category']] = imp_mode.fit_transform(df_messy[['category']])

print("After imputation:")
print(df_messy)
print(f"\nRemaining missing: {df_messy.isnull().sum().sum()}")

---
## 4 — Feature Creation

Create new features from existing ones. This is where domain knowledge shines.

Common strategies:
- **Polynomial features**: $x_1^2, x_1 \cdot x_2$
- **Ratios**: income/age, price/sqft
- **Date features**: day of week, month, is_weekend
- **Text features**: length, word count
- **Binning**: continuous → categorical (age → age_group)

In [ ]:
housing_df = fetch_california_housing(as_frame=True).frame

housing_df['rooms_per_household'] = housing_df['AveRooms'] / housing_df['AveOccup']
housing_df['bedrooms_ratio'] = housing_df['AveBedrms'] / housing_df['AveRooms']
housing_df['population_per_household'] = housing_df['Population'] / housing_df['HouseAge']

X_original = housing_df[['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']]
X_engineered = housing_df.drop('MedHouseVal', axis=1)
y_h = housing_df['MedHouseVal']

for name, X_data in [('Original features', X_original), ('+ Engineered features', X_engineered)]:
    X_tr, X_te, y_tr, y_te = train_test_split(X_data, y_h, test_size=0.2, random_state=42)
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_te_s = scaler.transform(X_te)
    
    rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    rf.fit(X_tr_s, y_tr)
    r2 = rf.score(X_te_s, y_te)
    print(f'{name:30s} → R² = {r2:.4f}')

In [ ]:
dates = pd.date_range('2024-01-01', periods=100, freq='D')
date_df = pd.DataFrame({'date': dates})

date_df['day_of_week'] = date_df['date'].dt.dayofweek
date_df['month'] = date_df['date'].dt.month
date_df['is_weekend'] = (date_df['day_of_week'] >= 5).astype(int)
date_df['day_of_year'] = date_df['date'].dt.dayofyear
date_df['quarter'] = date_df['date'].dt.quarter

print("Date feature engineering:")
date_df.head(10)

---
## 5 — Feature Selection

Too many features can hurt: noise, overfitting, slow training.  
Remove features that don't help.

Methods:
1. **Correlation analysis**: drop highly correlated features
2. **Mutual information**: statistical dependency with target
3. **Recursive Feature Elimination (RFE)**: iteratively drop least important features

In [ ]:
iris = load_iris()
X_iris = pd.DataFrame(iris.data, columns=iris.feature_names)
y_iris = iris.target

np.random.seed(42)
X_iris['noise_1'] = np.random.randn(150)
X_iris['noise_2'] = np.random.randn(150)
X_iris['noise_3'] = np.random.randn(150)

mi = mutual_info_classif(X_iris, y_iris, random_state=42)
mi_series = pd.Series(mi, index=X_iris.columns).sort_values(ascending=True)

plt.figure(figsize=(8, 5))
colors = ['#e74c3c' if 'noise' in name else '#3498db' for name in mi_series.index]
mi_series.plot(kind='barh', color=colors, edgecolor='black')
plt.xlabel('Mutual Information Score')
plt.title('Feature Selection — Noise features have near-zero MI')
plt.tight_layout()
plt.show()

In [ ]:
rfe = RFE(RandomForestClassifier(n_estimators=100, random_state=42), n_features_to_select=4)
rfe.fit(X_iris, y_iris)

selected = X_iris.columns[rfe.support_]
ranking = pd.Series(rfe.ranking_, index=X_iris.columns).sort_values()

print(f"Selected features: {list(selected)}")
print(f"\nFeature ranking:")
print(ranking)

all_score = cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42),
                            X_iris, y_iris, cv=5).mean()
selected_score = cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42),
                                 X_iris[selected], y_iris, cv=5).mean()

print(f"\nAll 7 features:    CV accuracy = {all_score:.4f}")
print(f"Selected 4 only:   CV accuracy = {selected_score:.4f}")

---
## 6 — sklearn Pipelines

Chain preprocessing + model into a single object. Benefits:
1. **No data leakage**: scaler is fit only on training data during CV
2. **Clean code**: one `fit()` / `predict()` does everything
3. **Easy deployment**: serialize one object

In [ ]:
simple_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=5000)),
])

scores = cross_val_score(simple_pipe, iris.data, iris.target, cv=5, scoring='accuracy')
print(f'Pipeline CV accuracy: {scores.mean():.4f} ± {scores.std():.4f}')

In [ ]:
np.random.seed(42)
df_full = pd.DataFrame({
    'age': np.random.randint(18, 70, 500).astype(float),
    'income': np.random.randint(20000, 150000, 500).astype(float),
    'education': np.random.choice(['high_school', 'bachelors', 'masters', 'phd'], 500),
    'city': np.random.choice(['NYC', 'LA', 'Chicago', 'Houston'], 500),
})

df_full.loc[np.random.choice(500, 30, replace=False), 'age'] = np.nan
df_full.loc[np.random.choice(500, 20, replace=False), 'income'] = np.nan

y_full = (df_full['income'].fillna(df_full['income'].median()) > 80000).astype(int)

numerical_features = ['age', 'income']
categorical_features = ['education', 'city']

from sklearn.preprocessing import OneHotEncoder

numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ('num', numerical_pipeline, numerical_features),
    ('cat', categorical_pipeline, categorical_features),
])

full_pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('model', RandomForestClassifier(n_estimators=100, random_state=42)),
])

scores = cross_val_score(full_pipeline, df_full, y_full, cv=5, scoring='accuracy')
print(f'Full pipeline CV accuracy: {scores.mean():.4f} ± {scores.std():.4f}')

---
## 7 — End-to-End: Raw Data → Pipeline → Model

Putting it all together on California Housing.

In [ ]:
housing = fetch_california_housing(as_frame=True)
df_h = housing.frame.copy()

df_h['rooms_per_hh'] = df_h['AveRooms'] * df_h['AveOccup']
df_h['bedrooms_ratio'] = df_h['AveBedrms'] / (df_h['AveRooms'] + 1e-8)
df_h['pop_density'] = df_h['Population'] / (df_h['AveOccup'] + 1e-8)

X_e2e = df_h.drop('MedHouseVal', axis=1)
y_e2e = df_h['MedHouseVal']

X_tr_e, X_te_e, y_tr_e, y_te_e = train_test_split(X_e2e, y_e2e, test_size=0.2, random_state=42)

e2e_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)),
])

e2e_pipeline.fit(X_tr_e, y_tr_e)
y_pred_e = e2e_pipeline.predict(X_te_e)

r2_final = r2_score(y_te_e, y_pred_e)
rmse_final = np.sqrt(np.mean((y_te_e - y_pred_e) ** 2))

print(f'Final model R²:   {r2_final:.4f}')
print(f'Final model RMSE: {rmse_final:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_te_e, y_pred_e, alpha=0.2, s=5)
axes[0].plot([0, 5], [0, 5], 'r--', linewidth=2)
axes[0].set_xlabel('Actual')
axes[0].set_ylabel('Predicted')
axes[0].set_title(f'Predictions vs Actual — R² = {r2_final:.3f}')

feat_imp = e2e_pipeline.named_steps['model'].feature_importances_
idx = np.argsort(feat_imp)
axes[1].barh(range(len(idx)), feat_imp[idx], color='steelblue', edgecolor='black')
axes[1].set_yticks(range(len(idx)))
axes[1].set_yticklabels(X_e2e.columns[idx])
axes[1].set_title('Feature Importances (including engineered)')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.show()

---
## Feature Engineering Cheat Sheet

| Task | Tool | Remember |
|------|------|----------|
| Scale numbers | StandardScaler, MinMaxScaler | Fit on train only! |
| Encode categories | One-hot (nominal), Ordinal (ordered) | drop_first avoids multicollinearity |
| Missing values | SimpleImputer (median/mode) | Consider adding a "was_missing" indicator |
| Create features | Ratios, polynomials, date parts | Domain knowledge is key |
| Select features | Mutual info, RFE, correlation | Drop noise → model often improves |
| Prevent leakage | sklearn Pipeline | All preprocessing inside the pipeline |

**The feature engineering workflow**:
1. Understand the data (EDA)
2. Handle missing values
3. Encode categoricals
4. Scale numericals
5. Create new features
6. Select best features
7. Wrap in a Pipeline
8. Cross-validate